# Notebook 01: Data and Metric Audit

**Purpose:** Verify that all data files load correctly and that both metric implementations behave as expected on synthetic examples.

Run this notebook first before any EDA or modelling. A clean pass here confirms the repo is in a working state.

In [ ]:
import warnings
import numpy as np
import pandas as pd

from src.utils.paths import check_raw_data
from src.data.load_data import (
    load_train, load_labels, load_test,
    load_target_pairs, load_test_labels, get_labels_for_lag,
)
from src.evaluation.metric import (
    spearman_sharpe_score, per_period_spearman,
    mean_row_pearson,
)

warnings.filterwarnings('ignore', category=UserWarning)
print('Imports OK')

## 1. Data file presence

In [ ]:
status = check_raw_data()
for name, present in status.items():
    print(f"  {'FOUND  ' if present else 'MISSING'}  {name}")

assert all(status.values()), "One or more data files are missing. Download from Kaggle."

## 2. Load and inspect all files

In [ ]:
train   = load_train()
labels  = load_labels()
test    = load_test()
pairs   = load_target_pairs()

print(f"\ntrain   shape : {train.shape}")
print(f"labels  shape : {labels.shape}")
print(f"test    shape : {test.shape}")
print(f"pairs   shape : {pairs.shape}")

print(f"\ntrain  date_id : {train['date_id'].min()} – {train['date_id'].max()}")
print(f"labels date_id : {labels['date_id'].min()} – {labels['date_id'].max()}")
print(f"test   date_id : {test['date_id'].min()} – {test['date_id'].max()}")

In [ ]:
# Confirm shapes match expected values from data_dictionary.md
assert train.shape  == (1961, 558), f"Unexpected train shape: {train.shape}"
assert labels.shape == (1961, 425), f"Unexpected labels shape: {labels.shape}"
assert test.shape   == (134,  559), f"Unexpected test shape: {test.shape}"
assert pairs.shape  == (424,  3),   f"Unexpected pairs shape: {pairs.shape}"
print('Shape assertions passed.')

In [ ]:
# Inspect target_pairs metadata
print('Lag distribution:')
print(pairs['lag'].value_counts().sort_index())
print()
print('Sample rows:')
pairs.head(8)

In [ ]:
# Test per-lag slicing
for lag in (1, 2, 3, 4):
    lag_labels = get_labels_for_lag(labels, pairs, lag)
    print(f"  lag={lag}: {lag_labels.shape}  targets: {lag_labels.shape[1] - 1}")

In [ ]:
# Load test labels for each lag
for lag in (1, 2, 3, 4):
    tl = load_test_labels(lag)
    print(f"  test_labels_lag_{lag}: {tl.shape}  date_id {tl['date_id'].min()}–{tl['date_id'].max()}")

## 3. Metric sanity checks

Verify both metric implementations on small synthetic examples with known expected values.

In [ ]:
# --- Spearman-Sharpe (official) ---

# Mostly positive periods → positive Sharpe
rng = np.random.default_rng(42)
T, N = 50, 20
true_data = pd.DataFrame(rng.standard_normal((T, N)))

# Good predictions: slightly noisy version of true
pred_good = true_data + 0.3 * pd.DataFrame(rng.standard_normal((T, N)))
score_good = spearman_sharpe_score(pred_good, true_data, nan_policy='ignore')

# Random predictions
pred_rand = pd.DataFrame(rng.standard_normal((T, N)))
score_rand = spearman_sharpe_score(pred_rand, true_data, nan_policy='ignore')

print(f'Spearman-Sharpe — good predictions : {score_good:.4f}  (expect > 0)')
print(f'Spearman-Sharpe — random predictions: {score_rand:.4f}  (expect ≈ 0)')
assert score_good > score_rand, "Good predictions should outscore random"

In [ ]:
# --- Pearson proxy ---
proxy_good = mean_row_pearson(pred_good, true_data, nan_policy='ignore')
proxy_rand = mean_row_pearson(pred_rand, true_data, nan_policy='ignore')

print(f'Pearson proxy  — good predictions : {proxy_good:.4f}  (expect > 0)')
print(f'Pearson proxy  — random predictions: {proxy_rand:.4f}  (expect ≈ 0)')

## 4. Missing value summary

In [ ]:
feature_cols = [c for c in train.columns if c != 'date_id']
target_cols  = [c for c in labels.columns if c != 'date_id']

train_missing  = train[feature_cols].isnull().sum().sum()
labels_missing = labels[target_cols].isnull().sum().sum()

print(f"Missing in train features : {train_missing:,}  "
      f"({100 * train_missing / (len(train) * len(feature_cols)):.1f}%)")
print(f"Missing in train labels   : {labels_missing:,}  "
      f"({100 * labels_missing / (len(labels) * len(target_cols)):.1f}%)")

# Top 10 features by missing count
top_missing = train[feature_cols].isnull().sum().sort_values(ascending=False).head(10)
print(f"\nTop 10 features by missing count:")
print(top_missing)

## 5. Summary

Fill in after running this notebook:

| Check | Result |
|-------|--------|
| All files present | ✓ |
| train shape | (1961, 558) |
| labels shape | (1961, 425) |
| test shape | (134, 559) |
| train missing % | TBD |
| labels missing % | TBD |
| Spearman-Sharpe (good pred) | TBD |
| Pearson proxy (good pred) | TBD |